In [4]:
import os
import warnings
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd
import igraph as ig
import leidenalg as la
import libpysal as lps
import esda
from tqdm import tqdm

from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import make_valid
from sklearn.metrics import normalized_mutual_info_score, fowlkes_mallows_score

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Mobility-flow-based hierarchy formulation

In [ ]:
# =========================================================
# 1. Consensus Leiden
# =========================================================
def _run_leiden_once(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    seed: int
):
    kwargs = dict(
        weights=weights,
        n_iterations=-1,
        seed=seed
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution
    return la.find_partition(G, partition_type, **kwargs)


def _multi_run_memberships(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    n_runs: int,
    seed0: int = 0
) -> List[List[int]]:
    membs = []
    for r in range(n_runs):
        part = _run_leiden_once(G, partition_type, resolution, weights, seed0 + r)
        membs.append(list(part.membership))
    return membs


def _coassoc_from_memberships(membs: List[List[int]]) -> np.ndarray:
    n = len(membs[0])
    P = np.zeros((n, n), dtype=np.float64)

    for m in membs:
        buckets: Dict[int, list] = {}
        for idx, c in enumerate(m):
            buckets.setdefault(c, []).append(idx)
        for idxs in buckets.values():
            idxs = np.asarray(idxs, dtype=int)
            P[np.ix_(idxs, idxs)] += 1.0

    P /= float(len(membs))
    np.fill_diagonal(P, 0.0)
    P = 0.5 * (P + P.T)
    return P


def _consensus_on_coassoc(
    P: np.ndarray,
    partition_type,
    resolution: float,
    threshold: Optional[float] = None
) -> Tuple[List[int], ig.Graph]:
    P_use = P.copy()
    if threshold is not None:
        P_use[P_use < threshold] = 0.0

    Gc = ig.Graph.Weighted_Adjacency(
        P_use.tolist(),
        mode="UNDIRECTED",
        attr="weight",
        loops=False
    )

    kwargs = dict(
        weights="weight",
        n_iterations=-1,
        seed=0
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution

    part = la.find_partition(Gc, partition_type, **kwargs)
    return list(part.membership), Gc


def _similarity(m1: List[int], m2: List[int], metric: str = "NMI") -> float:
    if metric.upper() == "NMI":
        return normalized_mutual_info_score(m1, m2)
    elif metric.upper() == "FMI":
        return fowlkes_mallows_score(m1, m2)
    else:
        raise ValueError("metric must be 'NMI' or 'FMI'.")


def iterative_consensus_leiden(
    G: ig.Graph,
    partition_type=la.ModularityVertexPartition,
    resolution: float = 1.0,
    weights: Optional[str] = "weight",
    n_runs: int = 100,
    seed0: int = 0,
    max_iter: int = 10,
    tol: float = 0.01,
    metric: str = "NMI",
    threshold: Optional[float] = None,
    return_history: bool = True
):
    if G is None or G.vcount() == 0:
        return {"membership": [], "history": [], "coassoc": None}

    membs0 = _multi_run_memberships(G, partition_type, resolution, weights, n_runs, seed0)
    P = _coassoc_from_memberships(membs0)
    memb_prev, Gc = _consensus_on_coassoc(P, partition_type, resolution, threshold)
    hist = [{"iter": 0, "similarity": np.nan, "n_comms": len(set(memb_prev))}]

    for it in range(1, max_iter + 1):
        membs = _multi_run_memberships(Gc, partition_type, resolution, "weight", n_runs, seed0 + it * 1000)
        P_next = _coassoc_from_memberships(membs)
        memb_next, Gc_next = _consensus_on_coassoc(P_next, partition_type, resolution, threshold)

        sim = _similarity(memb_prev, memb_next, metric=metric)
        hist.append({"iter": it, "similarity": sim, "n_comms": len(set(memb_next))})

        if 1.0 - sim < tol:
            return {
                "membership": memb_next,
                "history": hist if return_history else None,
                "coassoc": P_next
            }

        memb_prev, Gc, P = memb_next, Gc_next, P_next

    return {
        "membership": memb_prev,
        "history": hist if return_history else None,
        "coassoc": P
    }


# =========================================================
# 2. Geometry helpers
# =========================================================
def fill_holes(geom):
    if geom is None:
        return None

    geom = make_valid(geom)

    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)

    if geom.geom_type == "MultiPolygon":
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])

    return geom


def split_region_into_spatial_components(
    unit_region: gpd.GeoDataFrame,
    region_label: str,
    fill_component_holes: bool = False
) -> gpd.GeoDataFrame:
    """
    Divide the spatial unit corresponding to a community into multiple contiguous subregions based on spatial connectivity
    """
    if len(unit_region) == 0:
        return gpd.GeoDataFrame(
            columns=["component_id", "geometry"],
            geometry="geometry",
            crs=unit_region.crs
        )

    merged = unit_region.union_all()
    comps = gpd.GeoDataFrame(geometry=[merged], crs=unit_region.crs)
    comps = comps.explode(ignore_index=True, index_parts=False)

    if fill_component_holes:
        comps["geometry"] = comps["geometry"].apply(fill_holes)

    comps["component_id"] = [f"{region_label}_c{i+1}" for i in range(len(comps))]
    return comps[["component_id", "geometry"]].copy()


def assign_units_to_spatial_components(
    unit_region: gpd.GeoDataFrame,
    components: gpd.GeoDataFrame
) -> pd.DataFrame:
    """
    Mapping a cell to a subregion of a continuous space
    """
    if len(unit_region) == 0 or len(components) == 0:
        return pd.DataFrame(columns=["id", "component_id"])

    joined = gpd.sjoin(
        unit_region[["id", "geometry"]],
        components[["component_id", "geometry"]],
        how="inner",
        predicate="intersects"
    )

    joined = joined[["id", "component_id"]].drop_duplicates().copy()
    joined.index = range(len(joined))
    return joined


# =========================================================
# 3. Moran hotspot detection
# =========================================================
def identify_inflow_core(
    od: pd.DataFrame,
    unit: gpd.GeoDataFrame,
    crit_value: float = 0.05,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
    select_mode: str = "cumulative",   # "top1" or "cumulative"
    cum_share_threshold: float = 0.8
) -> gpd.GeoDataFrame:
    """
    Within the current region:
    1. Calculate the Local Moran's I using inflow density
    2. Select the "High-High" cells
    3. Merge them into hotspot clusters
    4. Retain the main clusters according to the rules

    select_mode:
        - "top1": Keep only the cluster with the highest inflow
        - "cumulative": Sort by inflow and keep clusters until the cumulative inflow reaches the threshold
    """
    if len(unit) == 0 or len(od) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    d = od.groupby("d_id", as_index=False)["flow"].sum()
    d.rename(columns={"d_id": "id", "flow": "inflow"}, inplace=True)

    unit2 = pd.merge(unit, d, on="id", how="left")
    unit2.fillna(0, inplace=True)
    unit2["inflow_den"] = unit2["inflow"] / unit2["area"]

    try:
        if use_queen:
            w = lps.weights.Queen.from_dataframe(unit2, use_index=True, silence_warnings=True)
        else:
            w = lps.weights.Rook.from_dataframe(unit2, use_index=True, silence_warnings=True)

        y = unit2["inflow_den"]
        lm = esda.Moran_Local(
            y,
            w,
            transformation="r",
            permutations=999,
            n_jobs=-1,
            seed=42
        )
        unit2["lisa"] = lm.get_cluster_labels(crit_value=crit_value)
    except Exception:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh = unit2.loc[unit2["lisa"] == "High-High"].copy()
    if len(unit_hh) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh.index = range(len(unit_hh))

    clusters = gpd.GeoDataFrame(geometry=[unit_hh.union_all()], crs=unit.crs)
    clusters = clusters.explode(ignore_index=True, index_parts=False)

    if fill_cluster_holes:
        clusters["geometry"] = clusters["geometry"].apply(fill_holes)

    clusters["cid"] = clusters.index + 1

    hhc = gpd.overlay(unit_hh, clusters, how="intersection", keep_geom_type=True)
    hhcg = hhc.groupby("cid", as_index=False)[["inflow", "area"]].sum()

    clusters = pd.merge(clusters, hhcg, on="cid", how="inner")
    if len(clusters) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    clusters = clusters.sort_values("inflow", ascending=False, ignore_index=True).copy()

    # -------- selection rule --------
    if select_mode == "top1":
        clusters = clusters.head(1).copy()

    elif select_mode == "cumulative":
        total_inflow = clusters["inflow"].sum()
        if total_inflow <= 0:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

        clusters["cum_share"] = clusters["inflow"].cumsum() / total_inflow
        keep_n = min((clusters["cum_share"] < cum_share_threshold).sum() + 1, len(clusters))
        clusters = clusters.head(keep_n).copy()

    else:
        raise ValueError("select_mode must be 'top1' or 'cumulative'")

    clusters.index = range(len(clusters))
    clusters.drop(columns=[c for c in ["cid", "cum_share"] if c in clusters.columns], inplace=True)

    if len(clusters) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    return clusters


# =========================================================
# 4. Graph helpers
# =========================================================
def induced_od(od: pd.DataFrame, node_ids: List[str]) -> pd.DataFrame:
    node_set = set(node_ids)
    out = od.loc[od["o_id"].isin(node_set) & od["d_id"].isin(node_set)].copy()
    out.index = range(len(out))
    return out


def build_graph_from_od(od_sub: pd.DataFrame) -> Optional[ig.Graph]:
    if len(od_sub) == 0:
        return None

    nodes = pd.unique(pd.concat([od_sub["o_id"], od_sub["d_id"]], axis=0))
    if len(nodes) < 2:
        return None

    G = ig.Graph.DataFrame(
        od_sub[["o_id", "d_id", "flow"]],
        directed=True,
        use_vids=False
    )
    G.es["weight"] = od_sub["flow"].tolist()
    return G


# =========================================================
# 5. Recursive hotspot + consensus community
# =========================================================
def recursive_hotspot_community(
    od_all: pd.DataFrame,
    unit_all: gpd.GeoDataFrame,
    node_ids: List[str],
    crit_value: float,
    cum_share_threshold: float,
    use_queen: bool,
    fill_cluster_holes: bool,
    n_runs: int,
    tol: float,
    resolution: float,
    min_units: int,
    level: int,
    path_prefix: str,
    unit_labels: pd.DataFrame,
    region_polygons: Dict[int, List[gpd.GeoDataFrame]],
    diagnostics: List[dict],
):
    """
    Logic: 
    1. Stop when the number of cells in the current region is insufficient; do not perform a hotspot split.
    2. The hotspot split is the primary step for identifying centers.
    3. After removing the center, if the number of remaining cells is insufficient, stop and do not perform a community split.
    4. Consensus communities only determine whether to partition.
       - n_comms >= 2: First partition by functional communities, then split by spatial connectivity, and continue recursively.
       - n_comms < 2: Do not partition; continue recursion on the whole.
    """

    unit_sub = unit_all.loc[unit_all["id"].isin(node_ids)].copy()
    unit_sub.index = range(len(unit_sub))
    od_sub = induced_od(od_all, node_ids)

    if len(unit_sub) == 0:
        return

    # -------------------------------------------------
    # Step 0. stop if too few units for Moran hotspot detection
    # -------------------------------------------------
    if len(unit_sub) < min_units:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1" if level == 1 else "cumulative",
            "reason": "too_few_units_for_moran"
        })
        return

    # -------------------------------------------------
    # Step 1. identify hotspot centers
    # -------------------------------------------------
    if level == 1:
        centers = identify_inflow_core(
            od_sub,
            unit_sub,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            select_mode="top1"
        )
        selection_mode = "top1"
    else:
        centers = identify_inflow_core(
            od_sub,
            unit_sub,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            select_mode="cumulative",
            cum_share_threshold=cum_share_threshold
        )
        selection_mode = "cumulative"

    if len(centers) == 0:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "no_hotspot"
        })
        return

    unit_center = gpd.overlay(
        unit_sub[["id", "geometry"]],
        centers[["geometry"]],
        how="intersection",
        keep_geom_type=True
    )
    center_ids = unit_center["id"].astype(str).unique().tolist()

    # Write to level only for cells that have not yet been assigned a value
    mask = unit_labels["id"].isin(center_ids)
    unit_labels.loc[mask & unit_labels["level"].isna(), "level"] = level

    # -------------------------------------------------
    # Step 2. remove centers
    # -------------------------------------------------
    rem_ids = [x for x in node_ids if x not in set(center_ids)]
    if len(rem_ids) == 0:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": 0,
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "all_center"
        })
        return

    # -------------------------------------------------
    # Step 2.5. stop if too few residual units for split
    # -------------------------------------------------
    if len(rem_ids) < min_units:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "too_few_units_for_split"
        })
        return

    od_rem = induced_od(od_all, rem_ids)
    G = build_graph_from_od(od_rem)

    if G is None:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "graph_none"
        })
        return

    # -------------------------------------------------
    # Step 3. consensus community on residual network
    # -------------------------------------------------
    res = iterative_consensus_leiden(
        G,
        partition_type=la.ModularityVertexPartition,
        resolution=resolution,
        weights="weight",
        n_runs=n_runs,
        tol=tol,
        metric="NMI",
        max_iter=10
    )

    membership = res["membership"]
    n_comms = len(set(membership))

    # -------------------------------------------------
    # Step 4. only one community: no partition, continue globally
    # -------------------------------------------------
    if n_comms < 2:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": n_comms,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "single_comm"
        })

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=rem_ids,
            crit_value=crit_value,
            cum_share_threshold=cum_share_threshold,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            min_units=min_units,
            level=level + 1,
            path_prefix=path_prefix,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )
        return

    # -------------------------------------------------
    # Step 5. multiple communities:
    # first split each community into spatially connected components,
    # then recurse on each connected component
    # -------------------------------------------------
    name_to_comm = dict(zip(G.vs["name"], membership))

    rem_df = pd.DataFrame({
        "id": list(name_to_comm.keys()),
        "comm_id": [name_to_comm[x] for x in name_to_comm.keys()]
    })

    if path_prefix == "":
        rem_df["comm_label"] = rem_df["comm_id"].astype(int).astype(str)
    else:
        rem_df["comm_label"] = path_prefix + "_" + rem_df["comm_id"].astype(int).astype(str)

    region_col = f"region_{level}"
    if region_col not in unit_labels.columns:
        unit_labels[region_col] = np.nan

    component_records = []
    polygon_records = []
    n_components_total = 0

    for comm_label in rem_df["comm_label"].dropna().unique().tolist():
        comm_ids = rem_df.loc[rem_df["comm_label"] == comm_label, "id"].astype(str).tolist()

        unit_comm = unit_all.loc[unit_all["id"].isin(comm_ids), ["id", "geometry"]].copy()
        unit_comm.index = range(len(unit_comm))

        comps = split_region_into_spatial_components(
            unit_comm,
            region_label=comm_label,
            fill_component_holes=False
        )

        if len(comps) == 0:
            continue

        n_components_total += len(comps)

        assign_df = assign_units_to_spatial_components(unit_comm, comps)
        if len(assign_df) == 0:
            continue

        component_records.append(assign_df)

        comp_poly = comps.copy()
        comp_poly.rename(columns={"component_id": "region_id"}, inplace=True)
        comp_poly["level"] = level
        polygon_records.append(comp_poly)

    diagnostics.append({
        "path": path_prefix if path_prefix != "" else "ROOT",
        "level": level,
        "n_units": len(node_ids),
        "n_centers": len(center_ids),
        "n_remaining": len(rem_ids),
        "n_comms": n_comms,
        "n_spatial_components": n_components_total,
        "continue_split": n_components_total >= 1,
        "selection_mode": selection_mode,
        "reason": "split"
    })

    if len(component_records) == 0:
        return

    rem_component_df = pd.concat(component_records, axis=0, ignore_index=True)
    rem_component_df = rem_component_df.rename(columns={"component_id": region_col})

    # Write the `region_level` tag in place
    mapping = dict(zip(rem_component_df["id"], rem_component_df[region_col]))
    unit_labels[region_col] = unit_labels["id"].map(mapping).combine_first(unit_labels[region_col])

    # Save the first two layers of polygons
    if level <= 2 and len(polygon_records) > 0:
        poly = pd.concat(polygon_records, axis=0, ignore_index=True)
        poly = gpd.GeoDataFrame(poly, geometry="geometry", crs=unit_all.crs)
        region_polygons.setdefault(level, []).append(poly)

    # Recursively enter each consecutive subregion
    child_regions = rem_component_df[region_col].dropna().unique().tolist()

    for rg in child_regions:
        child_ids = rem_component_df.loc[
            rem_component_df[region_col] == rg, "id"
        ].astype(str).tolist()

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=child_ids,
            crit_value=crit_value,
            cum_share_threshold=cum_share_threshold,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            min_units=min_units,
            level=level + 1,
            path_prefix=rg,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )


# =========================================================
# 6. Main pipeline
# =========================================================
def detect_hotspot_community_hierarchy(
    unit: gpd.GeoDataFrame,
    od: pd.DataFrame,
    crit_value: float = 0.05,
    cum_share_threshold: float = 0.8,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
    n_runs: int = 100,
    tol: float = 0.01,
    resolution: float = 1.0,
    min_units: int = 10
):
    """
    Returns: 
    - unit_result: The level, region_1, region_2, ... for each unit 
    - region_polygons_all: Polygons resulting from dissolving the first two layers 
    - diagnostics_df
    """

    unit = unit.copy()
    od = od.copy()

    unit["id"] = unit["id"].astype(str)
    od["o_id"] = od["o_id"].astype(str)
    od["d_id"] = od["d_id"].astype(str)

    if "area" not in unit.columns:
        unit["area"] = unit.geometry.area / 1e6

    keep_ids = pd.unique(pd.concat([od["o_id"], od["d_id"]], axis=0))
    unit = unit.loc[unit["id"].isin(keep_ids)].copy()
    unit.index = range(len(unit))

    unit_labels = unit[["id"]].copy()
    unit_labels["level"] = np.nan

    region_polygons = {}
    diagnostics = []

    root_ids = sorted(unit["id"].unique().tolist())

    recursive_hotspot_community(
        od_all=od,
        unit_all=unit,
        node_ids=root_ids,
        crit_value=crit_value,
        cum_share_threshold=cum_share_threshold,
        use_queen=use_queen,
        fill_cluster_holes=fill_cluster_holes,
        n_runs=n_runs,
        tol=tol,
        resolution=resolution,
        min_units=min_units,
        level=1,
        path_prefix="",
        unit_labels=unit_labels,
        region_polygons=region_polygons,
        diagnostics=diagnostics,
    )

    unit_result = pd.merge(unit, unit_labels, on="id", how="left")

    max_center_level = int(unit_result["level"].dropna().max()) if unit_result["level"].notna().any() else 0
    unit_result.loc[unit_result["level"].isna(), "level"] = max_center_level + 1
    unit_result["level"] = unit_result["level"].astype(int)

    region_polygons_all = {}
    for lv, polys in region_polygons.items():
        if len(polys) == 0:
            continue
        g = pd.concat(polys, axis=0, ignore_index=True)
        g = gpd.GeoDataFrame(g, geometry="geometry", crs=unit.crs)
        region_polygons_all[lv] = g

    diagnostics_df = pd.DataFrame(diagnostics)

    return unit_result, region_polygons_all, diagnostics_df


# =========================================================
# 7. Save outputs
# =========================================================
def save_hotspot_community_outputs(
    unit_result: gpd.GeoDataFrame,
    region_polygons_all: Dict[int, gpd.GeoDataFrame],
    diagnostics_df: pd.DataFrame,
    out_dir: str,
    prefix: str
):
    os.makedirs(out_dir, exist_ok=True)

    unit_result.to_file(os.path.join(out_dir, f"{prefix}_unit_hierarchy.shp"))

    if 1 in region_polygons_all:
        region_polygons_all[1].to_file(os.path.join(out_dir, f"{prefix}_region_level_1.shp"))

    if 2 in region_polygons_all:
        region_polygons_all[2].to_file(os.path.join(out_dir, f"{prefix}_region_level_2.shp"))

    diagnostics_df.to_csv(os.path.join(out_dir, f"{prefix}_diagnostics.csv"), index=False)


# =========================================================
# 8. Example usage
# =========================================================
def get_unit_name(city: str) -> str:
    if city in ["beijing", "shanghai", "shenzhen", "nanjing"]:
        return "grid_1k"
    elif city == "london":
        return "msoa"
    elif city in ["losangeles", "newyork"]:
        return "tract"
    else:
        raise ValueError(f"Unknown city: {city}")

In [ ]:
if __name__ == "__main__":
    for city in ["beijing","shanghai","shenzhen"]:
        unit_name = get_unit_name(city)

        od = pd.read_csv(f"D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv")
        od = od.loc[(od["o_id"] != od["d_id"]) & (od["flow"] > 0)].copy()
        od.index = range(len(od))

        unit = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{unit_name}.shp")
        unit["area"] = unit.geometry.area / 1e6
        unit = unit[["id", "area", "geometry"]].copy()
        unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
        unit.index = range(len(unit))

        unit_result, region_polygons_all, diagnostics_df = detect_hotspot_community_hierarchy(
            unit=unit,
            od=od,
            crit_value=0.05,
            cum_share_threshold=0.8,
            use_queen=False,
            fill_cluster_holes=True,
            n_runs=100,
            tol=0.01,
            resolution=1.0,
            min_units=10
        )

        save_hotspot_community_outputs(
            unit_result=unit_result,
            region_polygons_all=region_polygons_all,
            diagnostics_df=diagnostics_df,
            out_dir="D:/urban_hierarchy_congestion/results/hierarchy_formulation_results",
            prefix=city
        )
    
    for city in ["london","losangeles","newyork"]:
        unit_name = get_unit_name(city)

        od = pd.read_csv(f"D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv")
        od = od.loc[(od["o_id"] != od["d_id"]) & (od["flow"] > 0)].copy()
        od.index = range(len(od))

        unit = gpd.read_file(f"D:/urban_hierarchy_congestion/taz/{city}_{unit_name}.shp")
        unit["area"] = unit.geometry.area / 1e6
        unit = unit[["id", "area", "geometry"]].copy()
        unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
        unit.index = range(len(unit))

        unit_result, region_polygons_all, diagnostics_df = detect_hotspot_community_hierarchy(
            unit=unit,
            od=od,
            crit_value=0.05,
            cum_share_threshold=0.8,
            use_queen=True,
            fill_cluster_holes=True,
            n_runs=100,
            tol=0.01,
            resolution=1.0,
            min_units=10
        )

        save_hotspot_community_outputs(
            unit_result=unit_result,
            region_polygons_all=region_polygons_all,
            diagnostics_df=diagnostics_df,
            out_dir="D:/urban_hierarchy_congestion/results/hierarchy_formulation_results",
            prefix=city
        )

In [ ]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    if city in ['beijing','shanghai','shenzhen']:
        unit_name = 'grid_1k'
    elif city in ['london']:
        unit_name = 'mosa'
    else:
        unit_name = 'tract'
    poi = gpd.read_file(f'D:/urban_hierarchy_congestion/data/poi/{city}_poi.shp')
    unit = gpd.read_file(f'D:/urban_hierarchy_congestion/results/hierarchy_formulation_results/{city}_hierarchy.shp')
    poi.to_crs(crs=unit.crs,inplace=True)

    for i in ['commercial_service','education','health','government','industry','transport_hub']:
        unit_poi = gpd.sjoin(left_df=unit,right_df=poi.loc[(poi['type']==i),['type','geometry']],predicate='contains').groupby(['id'],as_index=False).size()
        unit = pd.merge(unit,unit_poi,on='id',how='left')
        unit.rename(columns={'size':i[:3]},inplace=True)
        unit.fillna(0,inplace=True)
    unit.to_file(f'D:/urban_hierarchy_congestion/results/hierarchy_formulation_results/{city}_hierarchy_poi.shp')

# Loubar-based hierarchy

In [ ]:
def compute_lorenz_curve(values):
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cum_nodes = np.arange(1, n + 1) / n
    cum_flows = np.cumsum(sorted_vals) / np.sum(sorted_vals)
    return cum_nodes, cum_flows

def gini(array):
    array = np.sort(np.array(array))
    n = len(array)
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * array)) / (n * np.sum(array))

def assign_hotspot_levels_auto(df,density_column):
    df = df.copy()
    remaining = df[['id', density_column]].copy()

    level = 1
    while remaining[density_column].mean() > 0:
        slope = remaining[density_column].max()/remaining[density_column].mean()

        if slope <= 1.001: 
            break

        threshold_frac = 1 - (1 / slope)
        threshold_frac = np.clip(threshold_frac, 0, 1)

        cutoff = int(np.ceil(len(remaining) * (1 - threshold_frac)))
        if cutoff <= 0:
            break

        top_ids = remaining.nlargest(cutoff, density_column)['id'].values
        df.loc[df['id'].isin(top_ids), 'level'] = level

        remaining = remaining[~remaining['id'].isin(top_ids)]

        level += 1
    
    df.loc[(df['level'].isnull()),'level'] = level
    df['level'] = df['level'].astype(int)

    return df

In [ ]:
def get_unit_name(city: str) -> str:
    if city in ["beijing", "shanghai", "shenzhen", "nanjing"]:
        return "grid_1k"
    elif city == "london":
        return "msoa"
    elif city in ["losangeles", "newyork"]:
        return "tract"
    else:
        raise ValueError(f"Unknown city: {city}")
    
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    unit_name = get_unit_name(city)
    od = pd.read_csv(f'D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv')
    d = od.groupby('d_id',as_index=False)['flow'].sum()
    d.rename(columns={'d_id':'id','flow':'inflow'},inplace=True)
    unit = gpd.read_file(f'D:/urban_hierarchy_congestion/data/taz/{city}_{unit_name}.shp')
    unit['area'] = unit.geometry.area/1e6
    unit = unit[["id", "area", "geometry"]].copy()
    unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
    unit.index = range(len(unit))
    
    unit = pd.merge(unit,d,on='id',how='left')
    unit.fillna(0,inplace=True)
    unit['inflowd'] = unit['inflow']/unit['area']
    unit = assign_hotspot_levels_auto(unit,'inflowd')
    unit.drop(columns=['inflow','inflowd'],inplace=True)
    unit['level'] = unit['level'].astype(int)
    unit.loc[(unit['level']>=4),'level'] = 4
    unit['region_1'] = '0'
    unit['region_2'] = '0'

    poi = gpd.read_file(f'D:/urban_hierarchy_congestion/data/poi/{city}_poi.shp')
    poi.to_crs(crs=unit.crs,inplace=True)

    for i in ['commercial_service','education','health','government','industry','transport_hub']:
        unit_poi = gpd.sjoin(left_df=unit,right_df=poi.loc[(poi['type']==i),['type','geometry']],predicate='contains').groupby(['id'],as_index=False).size()
        unit = pd.merge(unit,unit_poi,on='id',how='left')
        unit.rename(columns={'size':i[:3]},inplace=True)
        unit.fillna(0,inplace=True)
    unit.to_file(f'D:/urban_hierarchy_congestion/results/hierarchy_formulation_results/{city}_loubar_hierarchy_poi.shp')